# Genera reporte Mensual

In [53]:
# ==========================================
# 1. BIBLIOTECAS ESTÁNDAR DE PYTHON
# ==========================================
import os
import argparse
from datetime import datetime
import calendar
# ==========================================
# 2. PROCESAMIENTO DE DATOS Y GRÁFICOS
# ==========================================
import pandas as pd
import numpy as np
import pygmt
import matplotlib.pyplot as plt

# ==========================================
# 3. GENERACIÓN PDF (REPORTLAB)
# ==========================================
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Image as RLImage, 
    Table, TableStyle, PageBreak
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_JUSTIFY, TA_CENTER, TA_LEFT
from PIL import Image as PILImage

## 1. Parsear select.out
Parsea la cabecera hipocentral (Línea Tipo 1) del formato Nordic de SEISAN

In [2]:
# Función para  extraer informacion de archivo select.out (Parsear)
def parsear_nordic_select(archivo_select):
    """
    Parsea la cabecera hipocentral (Línea Tipo 1) del formato Nordic de SEISAN.
    """
    eventos = []
    
    if not os.path.exists(archivo_select):
        raise FileNotFoundError(f"No se encontro el archivo: {archivo_select}")

    with open(archivo_select, 'r', encoding='latin-1') as f:
        for linea in f:
            # Una linea tipo 1 tiene al menos 80 caracteres y la posicion 80 suele ser '1' o espacio en cabecera
            if len(linea) >= 70:
                indicador_tipo = linea[79:80] if len(linea) >= 80 else ' '
                if indicador_tipo in ['1', ' ']:
                    try:
                        # Extraer campos segun el estandar Nordic
                        anio = linea[1:5].strip()
                        mes = linea[6:8].strip()
                        dia = linea[8:10].strip()
                        hora = linea[11:13].strip() or '00'
                        minuto = linea[13:15].strip() or '00'
                        seg_str = linea[16:20].strip() or '0.0'
                        seg = int(float(seg_str))

                        lat_str = linea[23:30].strip()
                        lon_str = linea[30:38].strip()
                        prof_str = linea[38:43].strip()
                        mag_str = linea[55:59].strip()
                        tipo_mag = linea[59:60].strip()

                        if not (anio and mes and dia and lat_str and lon_str):
                            continue

                        fecha_dt = datetime(int(anio), int(mes), int(dia), int(hora), int(minuto), seg)
                        lat = float(lat_str)
                        lon = float(lon_str)
                        prof = float(prof_str) if prof_str else 0.0
                        mag = float(mag_str) if mag_str else 0.0

                        eventos.append({
                            "fecha_hora": fecha_dt,
                            "dia": int(dia),
                            "lat": lat,
                            "lon": lon,
                            "profundidad": prof,
                            "magnitud": mag,
                            "tipo_mag": tipo_mag if tipo_mag else "Mw"
                        })
                    except (ValueError, IndexError):
                        continue

    df = pd.DataFrame(eventos)
    return df

## 2. Figuras Histogramas y sismicidad diaria

In [107]:
# Funión paara genera los graficos (histogramas y sismicidad diaria) 

def format_grafico(axn):
    #Funcion para dar algunos formatos al grafico
    #Entrada: axes (axn),  etiqueta eje x (labelx), etiqueta eje y (labely)
    axn.spines['top'].set_visible(False)
    axn.spines['right'].set_visible(False)
    axn.spines['left'].set_visible(False)
    axn.tick_params(labelsize=16, labelright=True)
    #axn.get_yaxis().set_visible(False)
    
    axn.yaxis.grid(True, zorder=0, color='lightgray', linestyle='--')
    
    

def generar_graficos(df, out_dir="figuras_reporte"):
    os.makedirs(out_dir, exist_ok=True)
    
    # 1. Histograma de Magnitudes
    fig, ax=plt.subplots(figsize=(14,8), dpi=300)
    histo=ax.hist(df['magnitud'], bins=np.arange(1,9,1),edgecolor='k',alpha=1.0,color='darkorange',zorder=2)

    for n in range(len(histo[0])):
        if histo[0][n]!=0:
            ax.text(histo[1][n]+0.5,histo[0][n]+5,str(int(histo[0][n])),ha='center',fontsize=14, weight='bold')
        
    format_grafico(ax)        
    ax.set_xlabel("Magnitud",fontsize=14, fontweight='bold')
    ax.set_ylabel("Número de sismos", fontsize=14, fontweight='bold')

    ruta_hist = os.path.abspath(os.path.join(out_dir, "hist_magnitud.png"))
    fig.savefig(ruta_hist)
    plt.close(fig)

        
     # 2. Histograma Diario (Sismicidad acumulada en el mes)
    fig, ax = plt.subplots(figsize=(14, 8), dpi=300)
    conteo_diario = df.groupby('dia').size()
    bars = ax.bar(conteo_diario.index, conteo_diario.values, color="darkorange", edgecolor='k', width=0.8, zorder=2)
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, str(int(bar.get_height())),ha='center', color='k',
                fontsize=14)
    format_grafico(ax)
    #ax.set_title("Número de sismos por día", fontsize=14, fontweight="bold", pad=10)
    ax.set_xlabel("Día del mes", fontsize=14)
    #ax.grid(axis='y', linestyle='--', alpha=0.5)
    fig.tight_layout()
    
    ruta_diario = os.path.abspath(os.path.join(out_dir, "eventos_diarios.png"))
    fig.savefig(ruta_diario)
    plt.close(fig)

    return {
        "ruta_hist_mag": ruta_hist,
        "ruta_hist_diario": ruta_diario
    }

## 3. Genera mapa GMT con sismicidad por magnitud

In [102]:
# Función para generar de epicentros con PyGMT

def generate_map(df, out_dir="figuras_reporte"):
    
    """
    Genera el mapa de sismicidad mensual con PyGMT para Venezuela y areas adyacentes.
    """
    os.makedirs(out_dir, exist_ok=True)
    datos = df.copy()
    if 'mag' in datos.columns and 'magnitud' not in datos.columns:
        datos['magnitud'] = datos['mag']

    region = [-74.0, -58.0, 5.0, 14.0]

    # Separación por rangos de magnitud
    data_03 = datos[datos['magnitud'] < 3.0]
    data_34 = datos[(datos['magnitud'] >= 3.0) & (datos['magnitud'] < 4.0)]
    data_45 = datos[(datos['magnitud'] >= 4.0) & (datos['magnitud'] < 5.0)]
    data_5 = datos[datos['magnitud'] >= 5.0]

    # Paleta de colores topográfica
    cpt_relief = 'verde.cpt' if os.path.exists('verde.cpt') else 'geo'

    kwargs_grd = dict(
        grid='@earth_relief_15s',
        region=region,
        projection='M14.5c',
        cmap=cpt_relief,
    )

    fig = pygmt.Figure()

    # 1. Topografía sombreada
    fig.grdimage(shading=True, **kwargs_grd)

    # 2. Línea de costa, fronteras y ríos
    fig.coast(
        shorelines="0.5p,black",
        borders=['1/0.8p,gray20', '2/0.5p,gray40'],
        frame='a2f1',
        resolution='f',
        rivers='a/0.4p,deepskyblue4'
    )

    # 3. Rótulo geográfico
    fig.text(
        x=-68.0, 
        y=13.3, 
        text='Mar Caribe', 
        font='18p,Helvetica-Bold,white', 
        justify='MC'
    )

    # 4. Trazado de fallas activas (si existe el archivo en la ruta de trabajo)
    if os.path.exists("fallas2015.txt"):
        fig.plot(data="fallas2015.txt", pen="0.8p,red")

    # 5. Epicentros por rango de magnitud
    if not data_03.empty:
        fig.plot(x=data_03.lon, y=data_03.lat, style="c0.15c", fill='green', pen="0.3p,black")
    if not data_34.empty:
        fig.plot(x=data_34.lon, y=data_34.lat, style="c0.25c", fill='cyan', pen="0.3p,black")
    if not data_45.empty:
        fig.plot(x=data_45.lon, y=data_45.lat, style="c0.35c", fill='yellow', pen="0.4p,black")
    if not data_5.empty:
        fig.plot(x=data_5.lon, y=data_5.lat, style="c0.45c", fill='red', pen="0.5p,black")

    # 6. Leyenda de magnitudes
    ruta_leyenda = "legend_magnitud.txt"
      
    with pygmt.clib.Session() as session:
        session.call_module('gmtset', 'FONT_ANNOT_PRIMARY 7.5p')

    fig.legend(
        spec=ruta_leyenda,
        position='jBR+o0.15c+w3.6c/3.2c',
        box='+gwhite@20+p0.8p,gray30+r'
    )

   
    # 7. Guardar imagen final en alta resolución
    ruta_mapa = os.path.abspath(os.path.join(out_dir, "mapa_venezuela.png"))
    fig.savefig(ruta_mapa, dpi=300)
    #print(f"[OK] Mapa PyGMT generado: {out_png}")
    
    return {"ruta_mapa":ruta_mapa}


## 4. Crea informe PDF

In [134]:
# Funciones para generar el reporte en formato PDF

def agregar_cintillos(canvas, doc):
    """Inserta el cintillo superior e inferior en cada página generada."""

    canvas.saveState()
    ancho_pag, alto_pag = letter

    # Cintillo superior (ancho útil de 17.5 cm, centrado)
    cintillo_sup = "cintillo_superior.jpeg"
    if os.path.exists(cintillo_sup):
        canvas.drawImage(cintillo_sup, 2 * cm, alto_pag - 2.8 * cm, width=17.5 * cm, height=2.2 * cm, preserveAspectRatio=True)

    # Cintillo inferior
    cintillo_inf = "cintillo_inferior.png"
    if os.path.exists(cintillo_inf):
        canvas.drawImage(cintillo_inf, 2 * cm, 0.6 * cm, width=17.5 * cm, height=1.8 * cm, preserveAspectRatio=True)

    # Número de página en pie
    num_pag = canvas.getPageNumber()
    canvas.setFont("Helvetica", 8)
    canvas.setFillColor(colors.HexColor("#555555"))
    canvas.drawRightString(ancho_pag - 2 * cm, 0.8 * cm, f"Pág. {num_pag}")
    canvas.restoreState()

def construir_reporte_pdf(df, figuras):

    #  =================  Define variables a partir del dataframe ===========================
    total_sismos = len(df)
    
    sismo_mayor = df.loc[df['magnitud'].idxmax()] if total_sismos > 0 else None
    min_mag=df['magnitud'].min()
    max_mag=df['magnitud'].max()
    fecha = df_sismos.iloc[0,0]
    meses_es = {
        1: "enero", 2: "febrero", 3: "marzo", 4: "abril",
        5: "mayo", 6: "junio", 7: "julio", 8: "agosto",
        9: "septiembre", 10: "octubre", 11: "noviembre", 12: "diciembre"
    }
    year = fecha.year
    mes_nombre = meses_es[fecha.month]
    _, fin_day = calendar.monthrange(fecha.year, fecha.month)
    pdf_salida = "reporte_mensual"+'_'+mes_nombre+str(year)+".pdf"

    # =======================================================================================
    
    
    
    # Configuración de márgenes para dejar espacio a los cintillos
    doc = SimpleDocTemplate(
        pdf_salida,
        pagesize=letter,
        leftMargin=2 * cm,
        rightMargin=2 * cm,
        topMargin=3.2 * cm,
        bottomMargin=2.4 * cm
    )

    styles = getSampleStyleSheet()
    estilo_titulo = ParagraphStyle(
        'TituloReporte',
        parent=styles['Heading1'],
        fontName='Helvetica-Bold',
        fontSize=13,
        leading=16,
        textColor=colors.HexColor('#F54927'),
        alignment=1, # Centrado
        spaceAfter=4
    )
    estilo_subtitulo = ParagraphStyle(
        'Subtitulo',
        parent=styles['Normal'],
        fontName='Helvetica',
        fontSize=12,
        leading=13,
        textColor=colors.HexColor('#F54927'),
        alignment=1,
        spaceAfter=10
    )
    estilo_cuerpo = ParagraphStyle(
        'Cuerpo',
        parent=styles['Normal'],
        fontName='Helvetica',
        fontSize=10,
        alignment=TA_JUSTIFY,
        leading=13,
        textColor=colors.HexColor('#111111')
    )
    estilo_pie_fig = ParagraphStyle(
        'PieFig',
        parent=styles['Normal'],
        fontName='Helvetica',
        fontSize=10,
        leading=10,
        alignment=1,
        textColor=colors.HexColor('#111111')
    )

    story = []


    # ==================== PÁGINA 1 ====================
    story.append(Paragraph(f"REPORTE MENSUAL DE ACTIVIDAD SÍSMICA: 1° al {last_day} de {mes_nombre.upper()} {year}", estilo_titulo))
    #story.append(Paragraph(f"Período: {mes_nombre.upper()} {anio}", estilo_subtitulo))

    
    resumen_txt = (f"""
        Durante este mes se registraron <b>{total_sismos}</b> eventos sísmicos por el <b>Servicio Sismológico y de
        Alerta de Tsunamis (SSAT)</b> en la región comprendida entre la latitud 5° a 14° N y la longitud 58° a 74° O
        Las magnitudes de los sismos detectados variaron entre <b>{min_mag}</b> y <b>{max_mag}</b>. En la <b>Tabla 1</b>
        se muestra el número de eventos registrados según su ubicación geográfica y magnitud en tres regiones,
        occidente (longitud 69° a 74° O), centro (longitud 65° a 69°) y oriente (58° a 65° O) del país. En las 
        ilustraciones se muestra la distribución de los sismos por rangos de magnitud (Ilustración 1), número de
        sismos registrados por día (Ilustración 2) y el mapa con la ubicación epicentral discriminados por magnitud.""".replace("\n"," ")
    )
    story.append(Paragraph(resumen_txt, estilo_cuerpo))
    story.append(Spacer(1, 12))

    bins = [-74,-69,-65,-58]
    labels = ['Occidente','Centro','Oriente' ]
    bins_mag = [0,3,4,5,10]
    labels_mag = ['Mag < 3.0','3.0 <= Mag  < 4.0', '4.0 <= Mag <= 5.0', 'Mag >= 5.0']
    df_sismos['Región'] = pd.cut(df_sismos['lon'],bins=bins, labels = labels, right=False)
    df_sismos['Rango Mag']= pd.cut(df_sismos['magnitud'],bins=bins_mag, labels = labels_mag, right=False)
    grupos = df_sismos.groupby(['Región','Rango Mag'])['magnitud'].count()
    df_contingencia = grupos.unstack().fillna(0).astype(int)

   # Estilo de encabezado (negrita, centrado)
    estilo_th = ParagraphStyle(
        'HeaderStyle',
        parent=styles['Normal'],
        fontName='Helvetica-Bold',
        fontSize=8.5,
        leading=11,
        alignment=TA_CENTER,
        textColor=colors.HexColor('#111111')
    )
    
    # Estilo de etiquetas de región (Occidente, Centro, Oriente)
    estilo_lbl_region = ParagraphStyle(
        'RegionLabelStyle',
        parent=styles['Normal'],
        fontName='Helvetica-Bold',
        fontSize=8.5,
        leading=11,
        alignment=TA_CENTER,
        textColor=colors.HexColor('#111111')
    )
    
    # Estilo para los números
    estilo_td = ParagraphStyle(
        'CellNumberStyle',
        parent=styles['Normal'],
        fontName='Helvetica',
        fontSize=8.5,
        leading=11,
        alignment=TA_CENTER,
        textColor=colors.HexColor('#222222')
    )

    # 1. Cabecera en UNA SOLA FILA:
    # "Región" en la primera celda, seguida de los nombres de los rangos de magnitud
    fila_cabecera = [Paragraph("<b>Región</b>", estilo_th)]
    for col in df_contingencia.columns:
        # Reemplazar posibles secuencias LaTeX como \leq por el símbolo real ≤
        col_txt = str(col)
        fila_cabecera.append(Paragraph(f"<b>{col_txt}</b>", estilo_th))

    story.append(Paragraph("<b>Tabla 1.</b> Distribución de sismos registrados por región y rango de magnitud.", estilo_pie_fig))
    story.append(Spacer(1, 12))
    matriz_tabla = [fila_cabecera]

    # 2. Filas de datos
    for region, fila in df_contingencia.iterrows():
        fila_celdas = [Paragraph(str(region), estilo_lbl_region)]
        for valor in fila:
            fila_celdas.append(Paragraph(str(int(valor)), estilo_td))
        matriz_tabla.append(fila_celdas)

    # 3. Anchos de columnas ajustados (1 fila para Región y 4 para magnitudes)
    col_widths = [3.5 * cm] + [3.0 * cm] * len(df_contingencia.columns)

    tabla = Table(matriz_tabla, colWidths=col_widths)

    # 4. Estilos visuales: la línea divisoria ahora va debajo de la fila 0
    estilos = [
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ('TOPPADDING', (0, 0), (-1, -1), 4),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 4),

        # Línea horizontal ENCIMA de la cabecera (fila 0)
        ('LINEABOVE', (0, 0), (-1, 0), 1.0, colors.HexColor('#a0a0a0')),
        
        # Línea horizontal divisoria justo debajo de la cabecera (fila 0)
        ('LINEBELOW', (0, 0), (-1, 0), 0.8, colors.HexColor('#a0a0a0')),

        # Línea horizontal DEBAJO de la ÚLTIMA fila (fila -1)
        ('LINEBELOW', (0, -1), (-1, -1), 1.0, colors.HexColor('#a0a0a0')),
        
        # Fila sombreada suave en 'Centro' (ahora es el índice 2)
        ('BACKGROUND', (0, 2), (-1, 2), colors.HexColor('#f6f6f6')),
    ]
    tabla.setStyle(TableStyle(estilos))


    story.append(tabla)
    
 
    story.append(Spacer(1, 30))
    story.append(Paragraph("<b>Distribución de Magnitudes</b>", estilo_subtitulo))
    story.append(Spacer(1, 20))
    img_mag = RLImage(figuras["ruta_hist_mag"] ,width=14.5 * cm, height=8.0 * cm)
    story.append(img_mag)
    story.append(Paragraph("<b>Ilustración 1.</b> Distribución de magnitudes.", estilo_pie_fig))
    
    # ==================== PÁGINA 2 ====================
    story.append(PageBreak())

    story.append(Paragraph("<b>Número de sismos por día</b>", estilo_subtitulo))
    story.append(Spacer(1, 6))

    # Gráficos paralelos Matplotlib
    img_diario = RLImage(figuras["ruta_hist_diario"],width=14 * cm, height=8 * cm)
    
    #tabla_graficos = Table([[img_diario, img_mag]], colWidths=[8.5 * cm, 8.5 * cm])
    #tabla_graficos.setStyle(TableStyle([('ALIGN', (0,0), (-1,-1), 'CENTER')]))
    story.append(Spacer(1, 20))
    story.append(img_diario)
    story.append(Paragraph("<b>Ilustración 2.</b> Número de sismos por día.", estilo_pie_fig))
    story.append(Spacer(1, 20))

    story.append(Paragraph("<b>Mapa de epicentros</b>", estilo_subtitulo))
    # Mapa PyGMT

    story.append(Spacer(1, 20))
    if "ruta_mapa" in figuras and os.path.exists(figuras["ruta_mapa"]):
        with PILImage.open(figuras["ruta_mapa"]) as img:
            ancho_px, alto_px = img.size
        ancho=12
        story.append(RLImage(figuras["ruta_mapa"], width=12 * cm, height=alto_px/ancho_px*ancho * cm))
        story.append(Spacer(1, 10))
        story.append(Paragraph("<b>Ilustración 3.</b> Ubicación epicentral de los sismos registrados por el SSAT.", estilo_pie_fig))
            


    # Compilar aplicando cintillos en cada hoja
    doc.build(story, onFirstPage=agregar_cintillos, onLaterPages=agregar_cintillos)
    print(f"Creado {pdf_salida}")


## Crea las variables

In [129]:
df_sismos = parsear_nordic_select("select.out")
rutas_fig = generar_graficos(df_sismos)
ruta_map = generate_map(df_sismos)
rutas_fig.update(ruta_map)


In [135]:
# Crea reporte
construir_reporte_pdf(
        df=df_sismos,
        figuras=rutas_fig,
        )

Creado reporte_mensual_marzo2026.pdf


In [120]:
df_sismos

,fecha_hora,dia,lat,lon,profundidad,magnitud,tipo_mag,Región,Rango Mag
0,2026-03-01 03:09:11,1,7.974,-72.899,93.4,3.1,W,Occidente,3.0 <= Mag < 4.0
1,2026-03-01 03:25:39,1,10.657,-62.369,9.5,2.8,W,Oriente,Mag < 3.0
2,2026-03-01 07:22:01,1,8.685,-71.232,6.5,2.5,W,Occidente,Mag < 3.0
3,2026-03-01 10:12:51,1,7.760,-72.808,177.3,5.3,W,Occidente,Mag >= 5.0
4,2026-03-01 15:41:03,1,6.822,-73.108,154.7,3.7,W,Occidente,3.0 <= Mag < 4.0
...,...,...,...,...,...,...,...,...,...
214,2026-03-31 10:13:46,31,6.886,-73.035,158.4,2.8,W,Occidente,Mag < 3.0
215,2026-03-31 11:14:19,31,10.325,-61.862,23.1,3.9,W,Oriente,3.0 <= Mag < 4.0
216,2026-03-31 21:50:30,31,6.907,-73.236,143.7,2.4,W,Occidente,Mag < 3.0
217,2026-03-31 22:53:46,31,6.806,-73.155,151.0,2.7,W,Occidente,Mag < 3.0


In [127]:
fecha = df_sismos.iloc[0,0]
meses_es = {
        1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril",
        5: "Mayo", 6: "Junio", 7: "Julio", 8: "Agosto",
        9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre"
    }
year = fecha.year
meses_es[fecha.month].upper()

'MARZO'

In [138]:
import calendar
_, fin_day = calendar.monthrange(fecha.year, fecha.month)
fin_day

31